# ML Challenge: Customer Outcome Prediction (Luca)

## Ziel
Entwickeln Sie ein Klassifikationsmodell zur Vorhersage von `Target`.

### Aufgaben
1. Daten explorieren
2. Missing Values behandeln
3. Kategoriale Variablen kodieren
4. Mindestens 3 Modelle trainieren (nutzen auch Cross Validation) und base version erstellen
5. Accuracy, Precision, Recall und F1 vergleichen
6. Hyperparameter Tuning machen und GridCV
7. Accuracy, Precision, Recall und F1 vergleichen
8. Ergebnisse interpretieren


In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC


In [10]:
df = pd.read_csv('kunden.csv')  # Reads csv file into a dataframe
df.head()  # Outputs the first 5 rows of csv file


,PersonID,Target,ServiceLevel,FullName,Gender,Age,RelativesOnboard,Dependents,BookingID,PricePaid,Room,DepartureLocation
0,1,0,Basic,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,A
1,2,1,Premium,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,B
2,3,1,Basic,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,A
3,4,1,Premium,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,A
4,5,0,Basic,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,A


## 1. Exploration

In [12]:
print('Shape of Data: ')
print(df.shape)

print('\nAll columns and their datatypes in csv file: ')
print(df.dtypes)

print('\nAll columns that have missing values: ')
print(df.isna().sum())

print('\nStatistical conclusion of all data : ')
print('\n- shows things like count of all columns that have values : ')
df.describe(include='all')

Shape of Data: 
(891, 12)

All datatypes of data in csv file: 
PersonID               int64
Target                 int64
ServiceLevel             str
FullName                 str
Gender                   str
Age                  float64
RelativesOnboard       int64
Dependents             int64
BookingID                str
PricePaid            float64
Room                     str
DepartureLocation        str
dtype: object

All columns that have missing values: 
PersonID               0
Target                 0
ServiceLevel           0
FullName               0
Gender                 0
Age                  177
RelativesOnboard       0
Dependents             0
BookingID              0
PricePaid              0
Room                 687
DepartureLocation      2
dtype: int64

=== Statistische Zusammenfassung ===


,PersonID,Target,ServiceLevel,FullName,Gender,Age,RelativesOnboard,Dependents,BookingID,PricePaid,Room,DepartureLocation
count,891.000000,891.000000,891,891,891,714.000000,891.000000,891.000000,891,891.000000,204,889
unique,NaN,NaN,3,891,2,NaN,NaN,NaN,681,NaN,147,3
top,NaN,NaN,Basic,"Braund, Mr. Owen Harris",male,NaN,NaN,NaN,347082,NaN,G6,A
freq,NaN,NaN,491,1,577,NaN,NaN,NaN,7,NaN,4,644
mean,446.000000,0.383838,NaN,NaN,NaN,29.699118,0.523008,0.381594,NaN,32.204208,NaN,NaN
std,257.353842,0.486592,NaN,NaN,NaN,14.526497,1.102743,0.806057,NaN,49.693429,NaN,NaN
min,1.000000,0.000000,NaN,NaN,NaN,0.420000,0.000000,0.000000,NaN,0.000000,NaN,NaN
25%,223.500000,0.000000,NaN,NaN,NaN,20.125000,0.000000,0.000000,NaN,7.910400,NaN,NaN
50%,446.000000,0.000000,NaN,NaN,NaN,28.000000,0.000000,0.000000,NaN,14.454200,NaN,NaN
75%,668.500000,1.000000,NaN,NaN,NaN,38.000000,1.000000,0.000000,NaN,31.000000,NaN,NaN


In [35]:
print('- Target Distribution: ')
print(df['Target'].value_counts())

print('\n- Target Distribution (%): ')
print(f"Target=0: {df['Target'].value_counts()[0]} ({df['Target'].value_counts(normalize=True)[0] * 100:.1f}%)")
print(f"Target=1: {df['Target'].value_counts()[1]} ({df['Target'].value_counts(normalize=True)[1] * 100:.1f}%)")

print("------------------------")

print('\n- ServiceLevel Distribution: ')
print(df['ServiceLevel'].value_counts())

print('\n- ServiceLevel Distribution (%): ')
for service_level, count in df['ServiceLevel'].value_counts().items():
    print(f"ServiceLevel={service_level}: {count} "f"({df['ServiceLevel'].value_counts(normalize=True)[service_level] * 100:.1f}%)")

print("------------------------")

print('\n- Gender Distribution: ')
print(df['Gender'].value_counts())

print('\n- Gender Distribution (%): ')
for gender, count in df['Gender'].value_counts().items():
    print(f"Gender={gender}: {count} "f"({df['Gender'].value_counts(normalize=True)[gender] *100:.1f}%)")

print("------------------------")

print('\n- DepartureLocation Distribution: ')
print(df['DepartureLocation'].value_counts())

print('\n- DepartureLocation Distribution (%): ')
for location, count in df['DepartureLocation'].value_counts().items():
    percentage = df['DepartureLocation'].value_counts(normalize=True)[location] * 100
    print(f"DepartureLocation={location}: {count} ({percentage:.1f}%)")


- Target Distribution: 
Target
0    549
1    342
Name: count, dtype: int64

- Target Distribution (%): 
Target=0: 549 (61.6%)
Target=1: 342 (38.4%)
------------------------

- ServiceLevel Distribution: 
ServiceLevel
Basic       491
Premium     216
Standard    184
Name: count, dtype: int64

- ServiceLevel Distribution (%): 
ServiceLevel=Basic: 491 (55.1%)
ServiceLevel=Premium: 216 (24.2%)
ServiceLevel=Standard: 184 (20.7%)
------------------------

- Gender Distribution: 
Gender
male      577
female    314
Name: count, dtype: int64

- Gender Distribution (%): 
Gender=male: 577 (64.8%)
Gender=female: 314 (35.2%)
------------------------

- DepartureLocation Distribution: 
DepartureLocation
A    644
B    168
C     77
Name: count, dtype: int64

- DepartureLocation Distribution (%): 
DepartureLocation=A: 644 (72.4%)
DepartureLocation=B: 168 (18.9%)
DepartureLocation=C: 77 (8.7%)


## 2. Features und Target definieren

In [39]:
# Delete columns that have no value as a feature for the prediction:
# - PersonID: primary key -> no predictor
# - FullName: title can be extracted (Mr., Miss. ,etc..) -> rest of name is no predictor
# - BookingID: unique key -> no predictor
# - Room: too many missing values for it being a predictor

# Extract title from fullname with regex
df['Title'] = df['FullName'].str.extract(r',\s([A-Za-z]+)\.', expand=False)
print('Extracted titles:')
print(df['Title'].value_counts())

# Give rare title (<9 : 10% of all data rows) to "other"
title_counts = df['Title'].value_counts()
rare_titles = title_counts[title_counts < 9].index

def replace_rare_titles(x):
    if x in rare_titles:
        return 'Other'
    else:
        return x


df['Title'] = df['Title'].apply(replace_rare_titles)
print('\nCleaned titles:')
print(df['Title'].value_counts())

print('\nCleaned titles (%):')
for title, count in df['Title'].value_counts().items():
    print(f"Title={title}: {count} "f"({df['Title'].value_counts(normalize=True)[title] *100:.1f}%)")




Extracted titles:
Title
Mr          517
Miss        182
Mrs         125
Master       40
Dr            7
Rev           6
Major         2
Mlle          2
Col           2
Don           1
Mme           1
Ms            1
Lady          1
Sir           1
Capt          1
Jonkheer      1
Name: count, dtype: int64

Cleaned titles:
Title
Mr        517
Miss      182
Mrs       125
Master     40
Other      26
Name: count, dtype: int64

Cleaned titles (%):
Title=Mr: 517 (58.1%)
Title=Miss: 182 (20.4%)
Title=Mrs: 125 (14.0%)
Title=Master: 40 (4.5%)
Title=Other: 26 (2.9%)


In [40]:
# Define feature set
drop_cols = ['PersonID', 'FullName', 'BookingID', 'Room']
df_model = df.drop(columns=drop_cols)

# Split the data into the input data for later for the model (x, data without solution "target")
# and the output data with which the model should learn and get better (y, just the "target" column)
X = df_model.drop(columns=['Target'])
y = df_model['Target']

print('Features:', X.columns.tolist())
print('Shape X:', X.shape)
print('Shape y:', y.shape)

Features: ['ServiceLevel', 'Gender', 'Age', 'RelativesOnboard', 'Dependents', 'PricePaid', 'DepartureLocation', 'Title']
Shape X: (891, 8)
Shape y: (891,)


## 3. Preprocessing definieren

In [41]:
# Split features up into different kinds

# Numeric features: columns with numeric values, where distances are useful
numeric_features = ['Age', 'RelativesOnboard', 'Dependents', 'PricePaid']

# Ordinal features: column that has values with natural order but the distances can not be exactly
# measured
# ServiceLevel: Premium > Standard > Basic
ordinal_features = ['ServiceLevel']
ordinal_categories = [['Basic', 'Standard', 'Premium']]

# Categorical features: features without proper order (male cannot be > female for example)
categorical_features = ['Gender', 'DepartureLocation', 'Title']

print('Numeric features:', numeric_features)
print('Ordinal features:', ordinal_features)
print('Categorical features:', categorical_features)


Numeric features: ['Age', 'RelativesOnboard', 'Dependents', 'PricePaid']
Ordinal features: ['ServiceLevel']
Categorical features: ['Gender', 'DepartureLocation', 'Title']


In [42]:
from sklearn.preprocessing import OrdinalEncoder

# Preprocessing Pipelines

# Numeric transformer: fills up missing values (NaN) with median value of column
# Standard scaler brings up all numeric values from all numeric columns to same scale, so
# the model cant interpret some numeric values from another column as more "important"
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Ordinal transformer: fills up missing values (NaN) with most common value
# OrdinalEncoder transforms the ordinal values into numeric values, for example:
# Basic    ->  0
# Standard ->  1
# Premium  ->  2
ordinal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(categories=ordinal_categories))
])

# Categorical transformer: fills up missing values (NaN) with most common value
# OneHotEncoder: creates per catgeory own 0/1 column matrix, for example:
# Gender:
# male    ->  1  0
# female  ->  0  1

# DepartureLocation:
# A  ->  1  0  0
# B  ->  0  1  0
# C  ->  0  0  1

# if there is an unkwnon value in the predicition process, it will be coded as a complete 0 matrix
# => (handle_unknown='ignore')
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('ord', ordinal_transformer, ordinal_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

print('Preprocessor ready')

Preprocessor ready


## 4. Modelle trainieren (und verbessern)

## 5. Interpretation

In [ ]:
# TODO: Interpretieren Sie die Ergebnisse.
# - Welches Modell ist am besten?
# - Wo sind mögliche Overfitting-Risiken?
# - Welche Features könnten besonders wichtig sein?
